# Random Forest Feature Importance (LOOCV)

A simpler, single-model alternative to `Feature_Weighting.ipynb`. One Random Forest per sex, leave-one-out cross-validated (8 folds), reporting the top 10 genes by importance.

**Real data guard:** `USE_REAL_DATA` must stay `False` in this AI session. Only set `True` for your own local, offline run (see `CLAUDE.md`).

**Limitations:**
- Only 8 samples, thousands of genes — importances will be noisy. Treat this as a quick cross-check, not a replacement for the main pipeline.
- Male and female are kept separate.

In [ ]:
USE_REAL_DATA = True # keep False here. Only set True in your own local run, never inside an AI tool session, per CLAUDE.md.

import time
import pandas as pd

from random_forest_importance import run_loocv_random_forest

SEXES = ["female", "male"]
EXPECTED_GENE_COUNTS = {"female": 2881, "male": 4257}
DATA_DIR = "restructured"
TOP_N = 10

## Step 1. Load data and check shapes

In [ ]:
data = {}

for sex in SEXES:
    if USE_REAL_DATA:
        df = pd.read_csv(f"{DATA_DIR}/{sex}_expr.csv")
        with open(f"{DATA_DIR}/{sex}_filtered_gene_symbols.txt") as f:
            gene_cols = [line.strip() for line in f if line.strip()]
    else:
        df = pd.read_csv(f"{DATA_DIR}/{sex}_expr_synthetic.csv")
        gene_cols = [c for c in df.columns if c not in ("Sample", "Group", "Label")]

    assert df.shape[0] == 8, f"{sex}: expected 8 samples, got {df.shape[0]}"
    assert len(gene_cols) == EXPECTED_GENE_COUNTS[sex], (
        f"{sex}: expected {EXPECTED_GENE_COUNTS[sex]} genes, got {len(gene_cols)}"
    )
    assert (df["Group"].value_counts() == 4).all(), f"{sex}: expected a 4/4 Group split"
    assert (df["Label"] == (df["Group"] == "C26").astype(int)).all(), (
        f"{sex}: Label does not match Group for some samples"
    )

    data[sex] = {"X": df[gene_cols], "y": df["Label"]}
    print(f"{sex}: OK, {df.shape[0]} samples x {len(gene_cols)} genes, 4/4 Group split, Label matches Group")

## Step 2. Sanity check on toy data

One signal gene, four noise genes. The signal gene should rank #1.

In [ ]:
import numpy as np

toy_y = pd.Series([0, 0, 0, 0, 1, 1, 1, 1])

rng = np.random.default_rng(42)
toy_X = pd.DataFrame({
    "Signal": [5, 8, 10, 12, 120, 125, 130, 135],  # clean SHAM-low / C26-high split
    "Noise1": rng.uniform(0, 150, size=8),
    "Noise2": rng.uniform(0, 150, size=8),
    "Noise3": rng.uniform(0, 150, size=8),
    "Noise4": rng.uniform(0, 150, size=8),
})

toy_importance, toy_accuracy, _ = run_loocv_random_forest(toy_X, toy_y)
print(toy_importance)
print(f"toy LOOCV accuracy: {toy_accuracy:.0%}")

assert toy_importance.index[0] == "Signal", "Signal gene should rank #1 by mean importance"
print("PASS — signal gene ranked above all noise genes")

## Step 3. Run LOOCV Random Forest

Trains one Random Forest per held-out sample (8 folds), averages feature importance across folds, and reports LOOCV accuracy as a rough sanity check.

In [ ]:
importance_results = {}
accuracy_results = {}
fold_detail_results = {}

for sex in SEXES:
    t0 = time.time()
    importance, accuracy, fold_detail = run_loocv_random_forest(data[sex]["X"], data[sex]["y"])
    elapsed = time.time() - t0

    importance_results[sex] = importance
    accuracy_results[sex] = accuracy
    fold_detail_results[sex] = fold_detail

    print(f"--- {sex} ---  ({elapsed:.1f}s, {len(importance)} genes)")
    print(f"LOOCV accuracy: {accuracy:.0%} ({fold_detail['correct'].sum()}/{len(fold_detail)})")
    print(f"Top {TOP_N} genes by mean importance:")
    print(importance.head(TOP_N))
    print()

## Step 4. Save results

Saves the full importance ranking, per-fold predictions, and the top 10 genes for each sex.

In [ ]:
for sex in SEXES:
    importance = importance_results[sex]

    importance_path = f"{DATA_DIR}/{sex}_rf_importance.csv"
    top_genes_path = f"{DATA_DIR}/{sex}_rf_top{TOP_N}_genes.txt"
    fold_detail_path = f"{DATA_DIR}/{sex}_rf_loocv_folds.csv"

    importance.rename("MeanImportance").to_csv(importance_path, index_label="Gene")
    fold_detail_results[sex].to_csv(fold_detail_path, index=False)
    with open(top_genes_path, "w") as f:
        f.write("\n".join(importance.head(TOP_N).index.tolist()))

    print(f"{sex}: saved {len(importance)}-gene importance table -> {importance_path}")
    print(f"{sex}: saved top {TOP_N} genes -> {top_genes_path}")
    print(f"{sex}: saved {len(fold_detail_results[sex])}-fold LOOCV detail -> {fold_detail_path}")